# Learning a swing up policy using reinforcement learning (Basis function Q-learning)
by TimJanssen66

### General understanding
I'm using ...

## Table of contents

1. <a href="#Training-on-simulator">Training on simulator</a>
2. <a href="#Testing-on-simulator">Testing on simulator</a>
3. <a href="#Testing-on-setup">Testing on setup</a>
4. <a href="#Exercise-6:-Design-Assignment-Environment.">Exercise 6: Design Assignment Environment</a>


In [1]:
import numpy as np
import gymnasium as gym
import gym_unbalanced_disk
import matplotlib.pyplot as plt
import time

class CompatibilityWrapper(gym.Wrapper):
    """
    Translates old Gym API environments (like UnbalancedDisk) 
    to the modern Gymnasium API to prevent crash errors.
    """
    def reset(self, *, seed=None, options=None):
        # 1. Intercept the reset call and strip the unexpected kwargs
        obs = self.env.reset()
        
        # 2. Ensure the output is a tuple of (observation, info_dict)
        if not isinstance(obs, tuple):
            return obs, {}
        elif len(obs) == 2 and isinstance(obs[1], dict):
            return obs
        else:
            return obs, {}

    def step(self, action):
        result = self.env.step(action)
        
        # 1. If the old environment returns 4 variables (obs, reward, done, info)
        if len(result) == 4:
            obs, reward, done, info = result
            # Map 'done' to 'terminated', and set 'truncated' to False
            return obs, reward, done, False, info
            
        # 2. If it already returns 5, just pass it through
        return result

# Define discrete actions within the [-3, 3] constraint
# e.g., 5 actions: -3.0, -1.5, 0.0, 1.5, 3.0
DISCRETE_ACTIONS = np.linspace(-3.0, 3.0, 5)
NUM_ACTIONS = len(DISCRETE_ACTIONS)

In [2]:
def make_radial_basis_network(nvec, scale, bounds_low, bounds_high):
    """Creates a basis function mapping for the continuous state space."""
    Xvec = [np.linspace(l, h, num=ni) for l, h, ni in zip(bounds_low, bounds_high, nvec)]
    c_points = np.array(np.meshgrid(*Xvec))
    c_points = np.moveaxis(c_points, 0, -1)
    c_points = c_points.reshape((-1, c_points.shape[-1]))
    dx = np.array([X[1] - X[0] for X in Xvec])
    
    def basis_fun(obs):
        obs = np.array(obs)
        dis = (c_points - obs[None, :]) / dx[None, :]
        phi = np.exp(-0.5 * np.sum((dis / scale)**2, axis=1))
        return phi
    return basis_fun

## Training on simulator

In [9]:
def train_rbf_qlearning():
    # env = gym.make('unbalanced-disk-v0', dt=0.05)
    # env = gym.wrappers.TimeLimit(env, max_episode_steps=300)
    # 1. Instantiate the raw environment directly to bypass gym.make's strict version checkers
    base_env = gym_unbalanced_disk.UnbalancedDisk(dt=0.05)
    
    # 2. Wrap it with our new compatibility translator
    env = CompatibilityWrapper(base_env)
    
    # 3. Apply the time limit wrapper
    env = gym.wrappers.TimeLimit(env, max_episode_steps=300)
    
    # State bounds for the grid (angle wrapped to [-pi, pi], velocity approx [-40, 40])
    bounds_low = [-np.pi, -40.0]
    bounds_high = [np.pi, 40.0]
    
    # Create RBF network (e.g., 15x15 grid of basis functions)
    basis_fun = make_radial_basis_network(nvec=[15, 15], scale=1.0, bounds_low=bounds_low, bounds_high=bounds_high)
    
    # Initialize Theta: (N_basis_features x N_actions)
    obs, info = env.reset()
    n_basis = basis_fun(obs).shape[0]
    theta = np.zeros((n_basis, NUM_ACTIONS))
    
    # Hyperparameters
    epsilon = 0.15
    alpha = 0.05
    gamma = 0.98
    episodes = 500
    
    Q_val = lambda s: basis_fun(s) @ theta
    
    rewards_history = []

    print("Starting Training...")
    for ep in range(episodes):
        obs, info = env.reset()
        done = False
        total_reward = 0
        
        while not done:
            # Epsilon-greedy action selection
            if np.random.rand() < epsilon:
                action_idx = np.random.choice(NUM_ACTIONS)
            else:
                action_idx = np.random.choice(np.flatnonzero(Q_val(obs) == Q_val(obs).max()))
                
            # Map index to continuous voltage array required by the environment
            # u_applied = np.array([DISCRETE_ACTIONS[action_idx]], dtype=np.float32)
            # Map index to a native continuous voltage float required by SciPy's integrator
            u_applied = float(DISCRETE_ACTIONS[action_idx])
            
            obs_next, reward, terminated, truncated, info = env.step(u_applied)
            done = terminated or truncated
            total_reward += reward
            
            # Temporal Difference Update
            if terminated:
                target = reward
            else:
                target = reward + gamma * np.max(Q_val(obs_next))
                
            td_error = target - Q_val(obs)[action_idx]
            theta[:, action_idx] += alpha * td_error * basis_fun(obs)
            
            obs = obs_next
            
        rewards_history.append(total_reward)
        if (ep + 1) % 50 == 0:
            print(f"Episode {ep + 1}/{episodes} | Average Reward (last 50): {np.mean(rewards_history[-50:]):.2f}")

    # Save the trained weights
    np.save('rbf_theta.npy', theta)
    print("Training complete. Weights saved to 'rbf_theta.npy'")
    env.close()

if __name__ == "__main__":
    train_rbf_qlearning()

TypeError: UnbalancedDisk.reset() got an unexpected keyword argument 'options'

## Test on simulator

In [7]:
# (Include the exact same make_radial_basis_network function here)

def test_simulator():
    # env = gym.make('unbalanced-disk-v0', dt=0.05, render_mode='human')
    # 1. Instantiate raw environment with the render mode flag
    base_env = gym_unbalanced_disk.UnbalancedDisk(dt=0.05, render_mode='human')
    
    # 2. Wrap it with the compatibility translator
    env = CompatibilityWrapper(base_env)
    
    # Note: We do not need the TimeLimit wrapper here because our testing `while` 
    # loop will terminate automatically when the pendulum falls over, or you 
    # can manually stop it.
    
    bounds_low = [-np.pi, -40.0]
    bounds_high = [np.pi, 40.0]
    basis_fun = make_radial_basis_network(nvec=[15, 15], scale=1.0, bounds_low=bounds_low, bounds_high=bounds_high)
    
    # Load the trained weights
    try:
        theta = np.load('rbf_theta.npy')
    except FileNotFoundError:
        print("Error: 'rbf_theta.npy' not found. Run the training script first.")
        return

    Q_val = lambda s: basis_fun(s) @ theta

    obs, info = env.reset()
    done = False
    print("Testing on Simulator...")
    
    try:
        while not done:
            # Pure greedy policy (no epsilon exploration)
            action_idx = np.argmax(Q_val(obs))
            # u_applied = np.array([DISCRETE_ACTIONS[action_idx]], dtype=np.float32)
            u_applied = float(DISCRETE_ACTIONS[action_idx])
            
            obs, reward, terminated, truncated, info = env.step(u_applied)
            done = terminated or truncated
            
            env.render()
            time.sleep(1/24) # Real-time delay
    finally:
        env.close()

if __name__ == "__main__":
    test_simulator()

Testing on Simulator...


KeyboardInterrupt: 

## Test on setup

In [ ]:
# import hardware_unbalanced_disk # Replace with the actual hardware library provided in the lab
# (Include the exact same make_radial_basis_network function here)
def run_physical_hardware():
    # Replace this line with the hardware initialization command provided by the TA/documentation
    # env = hardware_unbalanced_disk.RealSetup(dt=0.05) 
    # 1. Instantiate the physical hardware connection (use the exact command your TA provides)
    base_env = hardware_unbalanced_disk.RealSetup(dt=0.05) 
    
    # 2. Wrap the hardware environment to protect your modern testing loop from old API crashes
    env = CompatibilityWrapper(base_env)
    
    bounds_low = [-np.pi, -40.0]
    bounds_high = [np.pi, 40.0]
    basis_fun = make_radial_basis_network(nvec=[15, 15], scale=1.0, bounds_low=bounds_low, bounds_high=bounds_high)
    
    # Load weights trained from the simulator
    theta = np.load('rbf_theta.npy')
    Q_val = lambda s: basis_fun(s) @ theta

    obs, info = env.reset()
    
    print("Starting Hardware Execution. Press Ctrl+C to emergency stop.")
    try:
        for step in range(300): # Run for a fixed number of steps (e.g., 15 seconds)
            # 1. State estimation (Read encoders)
            # Ensure the raw hardware state maps identically to simulator bounds
            
            # 2. Pure greedy policy decision
            action_idx = np.argmax(Q_val(obs))
            # u_applied = np.array([DISCRETE_ACTIONS[action_idx]], dtype=np.float32)
            u_applied = float(DISCRETE_ACTIONS[action_idx])
            
            # 3. Apply voltage to the real motor
            obs, reward, terminated, truncated, info = env.step(u_applied)
            
            # Hardware specific sleep loop might be required to ensure dt=0.05 is respected
            # time.sleep(0.05) 
            
            if terminated or truncated:
                break
                
    except KeyboardInterrupt:
        print("Emergency Stop Triggered.")
    finally:
        # Crucial: Always ensure the voltage is sent to 0V when shutting down the hardware
        # env.set_voltage(0.0)
        env.close()

if __name__ == "__main__":
    # run_physical_hardware()
    pass

# Exercise 6: Design Assignment Environment

In [ ]:
!pip install git+https://github.com/MaartenSchoukens/gym-unbalanced-disk@master
# in case of error try installing pyglet 
# !pip install pyglet


In [ ]:
import gymnasium as gym
import time
import gym_unbalanced_disk #is required
import numpy as np
env = gym.make('unbalanced-disk-v0',dt=0.05) #if this fails restart the kernel or use gym_unbalanced_disk.UnbalancedDisk
env = gym_unbalanced_disk.UnbalancedDisk(dt=0.05)
# I uncommented the last env = ..., Is that the right thing to do?


In [ ]:
#if you had error just run this cell again
observation, info = env.reset()
try:
    for i in range(200):
        u = env.action_space.sample() #random input
        observation, reward, terminated, truncated, info = env.step(u) 
        print(observation, reward)
        env.render()
        time.sleep(1/24)
        if terminated or truncated:
            observation, info = env.reset()
finally: #this will always run such to close the visualization
    env.close()

    

In [ ]:
Umax = 4

# c) Input sequence
# We create an open-loop input sequence that alternates maximum control effort.
# 20 steps * 0.05s = 1.0 seconds per swing direction.
ulist = np.concatenate([
    np.full(10, Umax),   # Swing right
    np.full(10, -Umax),  # Swing left
    np.full(10, Umax),   # Swing right (building momentum)
    np.full(10, -Umax),  # Swing left  (building momentum)
]) 
# This sequence actually swings back over the top and forth over the top
obs, info = env.reset()
try:
    for u in ulist:
        obs, reward, terminated, truncated, info = env.step(u)
        print(obs,reward)
        env.render()
        time.sleep(1/24)
        if terminated or truncated:
            obs, info = env.reset()
finally: #this will always run
    env.close()
    